In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torchvision.utils import save_image
from torch.utils.data import DataLoader
import os

In [2]:
# Create output folder
os.makedirs("generated_images", exist_ok=True)

In [3]:
# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [4]:
# Hyperparameters
latent_dim = 100
image_size = 28 * 28
batch_size = 64
epochs = 100
lr = 0.0002

In [5]:
# Transform MNIST images
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

In [6]:
# Load MNIST dataset
dataset = datasets.MNIST(
    root="data",
    train=True,
    transform=transform,
    download=True
)

100%|██████████| 9.91M/9.91M [00:01<00:00, 5.37MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 131kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.19MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 11.9MB/s]


In [7]:
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

In [8]:
# Generator model
class Generator(nn.Module):
    def __init__(self):
        super().__init__()

        self.model = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.LeakyReLU(0.2),

            nn.Linear(256, 512),
            nn.LeakyReLU(0.2),

            nn.Linear(512, 1024),
            nn.LeakyReLU(0.2),

            nn.Linear(1024, image_size),
            nn.Tanh()
        )

    def forward(self, z):
        img = self.model(z)
        img = img.view(img.size(0), 1, 28, 28)
        return img

In [9]:
# Discriminator model
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()

        self.model = nn.Sequential(
            nn.Linear(image_size, 512),
            nn.LeakyReLU(0.2),

            nn.Linear(512, 256),
            nn.LeakyReLU(0.2),

            nn.Linear(256, 1),
            nn.Sigmoid()
        )

    def forward(self, img):
        img = img.view(img.size(0), -1)
        validity = self.model(img)
        return validity

In [10]:
generator = Generator().to(device)
discriminator = Discriminator().to(device)

In [11]:
# Loss and optimizers
criterion = nn.BCELoss()

optimizer_G = optim.Adam(generator.parameters(), lr=lr)
optimizer_D = optim.Adam(discriminator.parameters(), lr=lr)

In [12]:
# Training loop
for epoch in range(epochs):
    for batch_idx, (real_images, _) in enumerate(loader):
        real_images = real_images.to(device)

        batch_size_current = real_images.size(0)

        # Real and fake labels
        real_labels = torch.ones(batch_size_current, 1).to(device)
        fake_labels = torch.zeros(batch_size_current, 1).to(device)

        # ---------------------
        # Train Discriminator
        # ---------------------
        z = torch.randn(batch_size_current, latent_dim).to(device)
        fake_images = generator(z)

        real_loss = criterion(discriminator(real_images), real_labels)
        fake_loss = criterion(discriminator(fake_images.detach()), fake_labels)

        d_loss = real_loss + fake_loss

        optimizer_D.zero_grad()
        d_loss.backward()
        optimizer_D.step()

        # ---------------------
        # Train Generator
        # ---------------------
        z = torch.randn(batch_size_current, latent_dim).to(device)
        generated_images = generator(z)

        g_loss = criterion(discriminator(generated_images), real_labels)

        optimizer_G.zero_grad()
        g_loss.backward()
        optimizer_G.step()

    print(
        f"Epoch [{epoch + 1}/{epochs}] "
        f"D Loss: {d_loss.item():.4f} "
        f"G Loss: {g_loss.item():.4f}"
    )

    # Save sample generated images after each epoch
    sample_z = torch.randn(25, latent_dim).to(device)
    sample_images = generator(sample_z)
    save_image(
        sample_images,
        f"generated_images/epoch_{epoch + 1}.png",
        nrow=5,
        normalize=True
    )

print("Training complete. Check the 'generated_images' folder.")

Epoch [1/100] D Loss: 0.1594 G Loss: 3.7971
Epoch [2/100] D Loss: 0.0164 G Loss: 6.3026
Epoch [3/100] D Loss: 0.0027 G Loss: 7.3544
Epoch [4/100] D Loss: 0.0001 G Loss: 14.2649
Epoch [5/100] D Loss: 0.0000 G Loss: 12.6061
Epoch [6/100] D Loss: 0.0000 G Loss: 13.4986
Epoch [7/100] D Loss: 0.0000 G Loss: 14.0974
Epoch [8/100] D Loss: 0.0012 G Loss: 7.9950
Epoch [9/100] D Loss: 0.0000 G Loss: 12.5346
Epoch [10/100] D Loss: 0.0000 G Loss: 13.7157
Epoch [11/100] D Loss: 0.0000 G Loss: 14.4358
Epoch [12/100] D Loss: 0.0033 G Loss: 11.3191
Epoch [13/100] D Loss: 0.0000 G Loss: 12.9857
Epoch [14/100] D Loss: 0.0097 G Loss: 7.2037
Epoch [15/100] D Loss: 0.0000 G Loss: 12.6314
Epoch [16/100] D Loss: 0.0000 G Loss: 14.1416
Epoch [17/100] D Loss: 0.0000 G Loss: 14.5354
Epoch [18/100] D Loss: 0.0000 G Loss: 15.1754
Epoch [19/100] D Loss: 0.0000 G Loss: 15.5314
Epoch [20/100] D Loss: 0.0000 G Loss: 16.3557
Epoch [21/100] D Loss: 0.0000 G Loss: 17.0194
Epoch [22/100] D Loss: 0.0000 G Loss: 17.6011
Ep